# Day 57 · Exercise 5: DeploymentChecker

**What you'll build:** Implement `DeploymentChecker` — a class that runs a suite of readiness checks (env vars, health endpoint) and produces a structured report. This is the automated pre-deployment checklist that replaces the 'did you remember to set SECRET_KEY?' conversation.

## Setup (provided)

In [ ]:
import os
from fastapi import FastAPI
from starlette.testclient import TestClient

# --- provided helpers (from earlier exercises) ---
def check_env_vars(required_vars: list[str], env: dict) -> tuple[bool, list[str]]:
    missing = [v for v in required_vars if v not in env or not env[v]]
    return len(missing) == 0, missing


## Your Implementation

In [ ]:
class DeploymentChecker:
    """Run a suite of deployment readiness checks and report results."""

    def __init__(self):
        self._checks: list[dict] = []

    def add_check(self, name: str, passed: bool, detail: str = "") -> None:
        """Record a named check result.

        Args:
            name:   Short identifier for the check (e.g. "health_endpoint").
            passed: True if the check passed.
            detail: Human-readable explanation (shown when check fails).
        """
        # TODO: append {"name": name, "passed": passed, "detail": detail}
        raise NotImplementedError

    def run_env_check(self, required_vars: list[str], env: dict) -> None:
        """Run check_env_vars and record result under name "env_vars".

        Args:
            required_vars: Variable names that must be present.
            env:           Dict to check (os.environ or test dict).
        """
        # TODO: ok, missing = check_env_vars(required_vars, env)
        # self.add_check("env_vars", ok,
        #     f"missing: {missing}" if not ok else "all present")
        raise NotImplementedError

    def run_health_check(self, app: FastAPI) -> None:
        """Call GET /health on app via TestClient and record result.

        Passes if status == 200 and response JSON has {"status": "ok"}.
        Records check under name "health_endpoint".
        """
        # TODO:
        # client = TestClient(app, raise_server_exceptions=False)
        # r = client.get("/health")
        # ok = r.status_code == 200 and r.json().get("status") == "ok"
        # self.add_check("health_endpoint", ok, f"status={r.status_code}")
        raise NotImplementedError

    def report(self) -> dict:
        """Return a summary report dict.

        Returns:
            {
              "passed": bool  — True iff ALL checks passed,
              "total":  int   — number of checks run,
              "checks": list[{"name", "passed", "detail"}]
            }
        """
        # TODO:
        # total = len(self._checks)
        # passed_count = sum(1 for c in self._checks if c["passed"])
        # return {"passed": passed_count == total, "total": total,
        #         "checks": self._checks}
        raise NotImplementedError


In [ ]:
class DeploymentChecker:
    def __init__(self):
        self._checks: list[dict] = []

    def add_check(self, name: str, passed: bool, detail: str = "") -> None:
        self._checks.append({"name": name, "passed": passed, "detail": detail})

    def run_env_check(self, required_vars: list[str], env: dict) -> None:
        ok, missing = check_env_vars(required_vars, env)
        self.add_check("env_vars", ok,
                       f"missing: {missing}" if not ok else "all present")

    def run_health_check(self, app: FastAPI) -> None:
        client = TestClient(app, raise_server_exceptions=False)
        r = client.get("/health")
        ok = r.status_code == 200 and r.json().get("status") == "ok"
        self.add_check("health_endpoint", ok, f"status={r.status_code}")

    def report(self) -> dict:
        total = len(self._checks)
        passed_count = sum(1 for c in self._checks if c["passed"])
        return {"passed": passed_count == total, "total": total,
                "checks": self._checks}


## Check Your Work

In [ ]:
def _run_checks():
    score = 0
    total = 5

    def _chk(n, ok, msg):
        nonlocal score
        print(f"  {'✅' if ok else '❌'} Check {n}: {msg}")
        if ok:
            score += 1

    try:
        checker = DeploymentChecker()
        checker.add_check("test_check", True, "all good")
    except NotImplementedError:
        for i in range(1, total + 1):
            print(f"  ❌ Check {i}: DeploymentChecker not implemented")
        print(f"\nScore: 0 / {total}")
        return

    try:
        rep = checker.report()
    except NotImplementedError:
        for i in range(1, total + 1):
            print(f"  ❌ Check {i}: DeploymentChecker.report not implemented")
        print(f"\nScore: 0 / {total}")
        return

    _chk(1, rep.get("passed") is True and rep.get("total") == 1,
         f"one passing check → passed=True, total=1 (got {rep})")

    # env check — all present
    try:
        c2 = DeploymentChecker()
        c2.run_env_check(["MODEL", "PORT"], {"MODEL": "llama3.2", "PORT": "8000"})
        rep2 = c2.report()
    except NotImplementedError:
        for i in range(2, 4):
            print(f"  ❌ Check {i}: run_env_check not implemented")
        rep2 = None

    if rep2 is not None:
        _chk(2, rep2.get("passed") is True,
             f"env check passes when all vars present (got {rep2})")
        c3 = DeploymentChecker()
        c3.run_env_check(["MODEL", "SECRET_KEY"], {"MODEL": "llama3.2"})
        rep3 = c3.report()
        _chk(3, rep3.get("passed") is False,
             f"env check fails when SECRET_KEY missing (got {rep3})")

    # health check
    try:
        from datetime import datetime
        health_app = FastAPI()

        @health_app.get("/health")
        def _h():
            return {"status": "ok", "timestamp": datetime.utcnow().isoformat(), "version": "1.0"}

        c4 = DeploymentChecker()
        c4.run_health_check(health_app)
        rep4 = c4.report()
    except NotImplementedError:
        print(f"  ❌ Check 4: run_health_check not implemented")
        rep4 = None

    if rep4 is not None:
        _chk(4, rep4.get("passed") is True,
             f"health check passes on /health app (got {rep4})")

    # combined: one pass + one fail → not passed
    c5 = DeploymentChecker()
    c5.add_check("ok_check", True)
    c5.add_check("fail_check", False, "something wrong")
    rep5 = c5.report()
    _chk(5, rep5.get("passed") is False and rep5.get("total") == 2,
         f"mix of pass/fail → passed=False, total=2 (got {rep5})")

    print(f"\nScore: {score} / {total}")
    if score == total:
        print("🎉 Exercise complete!")

_run_checks()


## Bonus Challenge

Add a `run_cors_check(app, allowed_origin)` method to `DeploymentChecker`. It should use TestClient to send a GET /health with an `Origin: {allowed_origin}` header and check that the response includes `access-control-allow-origin` (which means CORSMiddleware is configured). Record under name 'cors'. This tests that the CORS config will work for the expected frontend origin.

## Solution

<details>
<summary>Show solution</summary>

```python
class DeploymentChecker:
    def __init__(self):
        self._checks: list[dict] = []

    def add_check(self, name: str, passed: bool, detail: str = "") -> None:
        self._checks.append({"name": name, "passed": passed, "detail": detail})

    def run_env_check(self, required_vars: list[str], env: dict) -> None:
        ok, missing = check_env_vars(required_vars, env)
        self.add_check("env_vars", ok,
                       f"missing: {missing}" if not ok else "all present")

    def run_health_check(self, app: FastAPI) -> None:
        client = TestClient(app, raise_server_exceptions=False)
        r = client.get("/health")
        ok = r.status_code == 200 and r.json().get("status") == "ok"
        self.add_check("health_endpoint", ok, f"status={r.status_code}")

    def report(self) -> dict:
        total = len(self._checks)
        passed_count = sum(1 for c in self._checks if c["passed"])
        return {"passed": passed_count == total, "total": total,
                "checks": self._checks}
```

**Why this works:** `DeploymentChecker` accumulates check results in a list
rather than printing them immediately — this separates data from display and
makes the results programmable (you can loop over `report()["checks"]` and
format however you want). `run_env_check` and `run_health_check` are convenience
methods that call `add_check` with the right name and detail. The `report()`
method aggregates: `passed` is only True when ALL checks pass — one failure
means the deployment is not ready.

</details>